In [ ]:
import time
import math
import random

from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit_algorithms import Grover, AmplificationProblem
from qiskit.circuit.library import PhaseOracle

import matplotlib.pyplot as plt


backend = AerSimulator()
results = []

print("Running Grover Simulation Scaling Experiment...\n")

# ----------------------------------
# Main Experiment Loop
# ----------------------------------
for n_qubits in range(2, 27):  # Increased range (up to 25 qubits)

    start_time = time.time()

    # Variable names: a, b, c, ...
    variables = [chr(ord('a') + i) for i in range(n_qubits)]

    # random target instead of 000...1
    target_int = random.randint(0, 2**n_qubits - 1)
    target = format(target_int, f"0{n_qubits}b")

    # Build Boolean oracle expression
    expr = []
    for var, bit in zip(variables, target):
        if bit == "1":
            expr.append(var)
        else:
            expr.append(f"~{var}")

    oracle_expression = " & ".join(expr)

    # Create oracle
    oracle = PhaseOracle(oracle_expression)

    # Define problem
    problem = AmplificationProblem(oracle, is_good_state=target)

    # Grover iterations
    N = 2 ** n_qubits
    iterations = int((math.pi / 4) * math.sqrt(N))

    grover = Grover(iterations=iterations)

    # Build circuit
    circuit = grover.construct_circuit(problem)
    circuit.measure_all()

    # Transpile for simulator
    transpiled_circuit = transpile(circuit, backend)

    # get circuit complexity info
    depth = transpiled_circuit.depth()
    gates = transpiled_circuit.size()

    #  print circuit for small cases only
    if n_qubits <= 4:
        print(f"\nTranspiled Circuit for {n_qubits} qubits:")
        print(transpiled_circuit)

    # Run simulation
    job = backend.run(transpiled_circuit, shots=1024)
    counts = job.result().get_counts()

    end_time = time.time()
    runtime = end_time - start_time

    # Fix bit ordering
    most_likely = max(counts, key=counts.get)[::-1]
    success_prob = counts.get(target[::-1], 0) / 1024

    # Store results
    results.append({
        "qubits": n_qubits,
        "states": N,
        "iterations": iterations,
        "target": target,
        "result": most_likely,
        "success_prob": success_prob,
        "runtime": runtime,
        "depth": depth,
        "gates": gates
    })

    print(f"{n_qubits} qubits ({N} states):")
    print(f"  Target: {target}")
    print(f"  Found:  {most_likely}")
    print(f"  Success Prob: {success_prob:.3f}")
    print(f"  Iterations: {iterations}")
    print(f"  Runtime: {runtime:.4f} sec")
    print(f"  Circuit Depth: {depth}")
    print(f"  Gate Count: {gates}\n")

# ----------------------------------
# Final Results
# ----------------------------------
print("\nFinal Results Summary:")
for r in results:
    print(r)

# ----------------------------------
# Graphs
# ----------------------------------
qubits = [r["qubits"] for r in results]
runtimes = [r["runtime"] for r in results]
success = [r["success_prob"] for r in results]
depths = [r["depth"] for r in results]

# Runtime plot
plt.figure()
plt.plot(qubits, runtimes, marker='o')
plt.xlabel("Number of Qubits")
plt.ylabel("Runtime (seconds)")
plt.title("Runtime vs Qubits (Simulation Scaling)")
plt.show()

# Success probability plot
plt.figure()
plt.plot(qubits, success, marker='o')
plt.xlabel("Number of Qubits")
plt.ylabel("Success Probability")
plt.title("Success Probability vs Qubits")
plt.show()

# Circuit depth plot
plt.figure()
plt.plot(qubits, depths, marker='o')
plt.xlabel("Number of Qubits")
plt.ylabel("Circuit Depth")
plt.title("Circuit Depth vs Qubits")
plt.show()

Running Grover Simulation Scaling Experiment...



C:\Users\lyche\AppData\Local\Temp\ipykernel_10564\3447655755.py:43: DeprecationWarning: The class ``qiskit.circuit.library.phase_oracle.PhaseOracle`` is deprecated as of Qiskit 2.2. It will be removed in Qiskit 3.0. Use the class qiskit.circuit.library.PhaseOracleGate instead.
  oracle = PhaseOracle(oracle_expression)



Transpiled Circuit for 2 qubits:
        ┌─────────┐     ┌──────────┐      ┌───────────┐ ░ ┌─┐   
   q_0: ┤ U2(0,0) ├─■───┤ U2(-π,0) ├───■──┤ U2(-π,-π) ├─░─┤M├───
        ├─────────┤ │ ┌─┴──────────┴┐┌─┴─┐└─┬────────┬┘ ░ └╥┘┌─┐
   q_1: ┤ U2(0,0) ├─■─┤ U3(π,-π,-π) ├┤ X ├──┤ U1(-π) ├──░──╫─┤M├
        └─────────┘   └─────────────┘└───┘  └────────┘  ░  ║ └╥┘
meas: 2/═══════════════════════════════════════════════════╩══╩═
                                                           0  1 
2 qubits (4 states):
  Target: 00
  Found:  00
  Success Prob: 1.000
  Iterations: 1
  Runtime: 2.5565 sec
  Circuit Depth: 6
  Gate Count: 10



C:\Users\lyche\AppData\Local\Temp\ipykernel_10564\3447655755.py:43: DeprecationWarning: The class ``qiskit.circuit.library.phase_oracle.PhaseOracle`` is deprecated as of Qiskit 2.2. It will be removed in Qiskit 3.0. Use the class qiskit.circuit.library.PhaseOracleGate instead.
  oracle = PhaseOracle(oracle_expression)



Transpiled Circuit for 3 qubits:
           ┌───┐        ┌─────────┐       ┌───────────┐     ┌─────────┐       »
   q_0: ───┤ H ├────■───┤ U2(0,0) ├────■──┤ U2(-π,-π) ├─■───┤ U2(0,0) ├────■──»
           ├───┤    │   ├─────────┤    │  ├───────────┤ │   ├─────────┤    │  »
   q_1: ───┤ H ├────■───┤ U2(0,0) ├────■──┤ U2(-π,-π) ├─■───┤ U2(0,0) ├────■──»
        ┌──┴───┴──┐ │ ┌─┴─────────┴─┐┌─┴─┐├───────────┤ │ ┌─┴─────────┴─┐┌─┴─┐»
   q_2: ┤ U2(0,0) ├─■─┤ U3(π,-π,-π) ├┤ X ├┤ U3(π,0,0) ├─■─┤ U3(π,-π,-π) ├┤ X ├»
        └─────────┘   └─────────────┘└───┘└───────────┘   └─────────────┘└───┘»
meas: 3/══════════════════════════════════════════════════════════════════════»
                                                                              »
«        ┌───────────┐ ░ ┌─┐      
«   q_0: ┤ U2(-π,-π) ├─░─┤M├──────
«        ├───────────┤ ░ └╥┘┌─┐   
«   q_1: ┤ U2(-π,-π) ├─░──╫─┤M├───
«        └─┬────────┬┘ ░  ║ └╥┘┌─┐
«   q_2: ──┤ U1(-π) ├──░──╫──╫─┤M├
«          └────────┘  ░  ║  ║ └╥┘
«

...

C:\Users\lyche\AppData\Local\Temp\ipykernel_10564\3447655755.py:43: DeprecationWarning: The class ``qiskit.circuit.library.phase_oracle.PhaseOracle`` is deprecated as of Qiskit 2.2. It will be removed in Qiskit 3.0. Use the class qiskit.circuit.library.PhaseOracleGate instead.
  oracle = PhaseOracle(oracle_expression)


20 qubits (1048576 states):
  Target: 01001011111010101110
  Found:  01001011111010101110
  Success Prob: 1.000
  Iterations: 804
  Runtime: 160.0643 sec
  Circuit Depth: 3218
  Gate Count: 33808



C:\Users\lyche\AppData\Local\Temp\ipykernel_10564\3447655755.py:43: DeprecationWarning: The class ``qiskit.circuit.library.phase_oracle.PhaseOracle`` is deprecated as of Qiskit 2.2. It will be removed in Qiskit 3.0. Use the class qiskit.circuit.library.PhaseOracleGate instead.
  oracle = PhaseOracle(oracle_expression)


21 qubits (2097152 states):
  Target: 100000011010111100011
  Found:  100000011010111100011
  Success Prob: 1.000
  Iterations: 1137
  Runtime: 304.0253 sec
  Circuit Depth: 4550
  Gate Count: 50069



C:\Users\lyche\AppData\Local\Temp\ipykernel_10564\3447655755.py:43: DeprecationWarning: The class ``qiskit.circuit.library.phase_oracle.PhaseOracle`` is deprecated as of Qiskit 2.2. It will be removed in Qiskit 3.0. Use the class qiskit.circuit.library.PhaseOracleGate instead.
  oracle = PhaseOracle(oracle_expression)
